# Fleetwise Seller Hub: follow one deal through every layer

This notebook follows **Jordan Rivera's top open deal** (`0060g000000037YAAQ`) from the raw Salesforce
export file in the UC volume through bronze, silver and gold (Lakeflow Declarative Pipeline), the
GenAI call insights (`ai_query`), the OpenJev deal score, the next-best-action queue, the Lakebase
Postgres row the app reads, a Genie answer, and finally the app's own REST API.

Every output is printed as text so it can be read without running anything.

**How it ran:** executed with `jupyter nbconvert --execute` on a laptop acting as a *client* of the
live workspace: all SQL runs on the workspace's serverless SQL warehouse, files come from the UC volume,
Postgres is the Lakebase Autoscaling endpoint, Genie and the app are called over their REST APIs. Auth is
the developer's own OAuth (CLI profile); no tokens are stored. Data is synthetic (no real customer).

In [1]:
import json, logging, sys, time
from pathlib import Path
import pandas as pd, requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import Disposition, Format, StatementParameterListItem

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "tests" / "integration")]
from _lakebase_helpers import connect
from lakebase import setup_lakebase as lb

PROFILE, WAREHOUSE = "fevm-serverless-stable-s1omq8", "9a6db1d12e2ee6d3"
CATALOG, SCHEMA = "serverless_stable_s1omq8_catalog", "seller_superapp"
DEAL = "0060g000000037YAAQ"
w = WorkspaceClient(profile=PROFILE)
logging.getLogger("databricks.sdk").setLevel(logging.ERROR)  # quiet SDK endpoint-probe warnings
pd.set_option("display.width", 200, "display.max_colwidth", 120)

def q(sql, **params):
    r = w.statement_execution.execute_statement(
        statement=sql, warehouse_id=WAREHOUSE, catalog=CATALOG, schema=SCHEMA, wait_timeout="50s",
        disposition=Disposition.INLINE, format=Format.JSON_ARRAY,
        parameters=[StatementParameterListItem(name=k, value=str(v)) for k, v in params.items()] or None)
    while r.status.state.value in ("PENDING", "RUNNING"):
        time.sleep(2); r = w.statement_execution.get_statement(r.statement_id)
    assert r.status.state.value == "SUCCEEDED", r.status.error
    cols = [c.name for c in r.manifest.schema.columns]
    return pd.DataFrame(r.result.data_array or [], columns=cols)

def show(df, transpose=False):
    print((df.T if transpose else df).to_markdown())

print("workspace user:", w.current_user.me().user_name)
print(q("SELECT current_user() AS sql_user, current_catalog() AS catalog, current_schema() AS schema").to_markdown(index=False))

workspace user: will.taff@databricks.com


| sql_user                 | catalog                          | schema          |
|:-------------------------|:---------------------------------|:----------------|
| will.taff@databricks.com | serverless_stable_s1omq8_catalog | seller_superapp |


## 1. Raw file (UC volume, Salesforce-API-shaped JSON export)
The bronze row records which file it came from (`_metadata.file_path`).

In [2]:
src = q("SELECT _source_file FROM bronze_opportunity WHERE Id = :id ORDER BY batch DESC LIMIT 1", id=DEAL)._source_file[0]
print("source file:", src)
lines = w.files.download(src).contents.read().decode().splitlines()
raw = next(json.loads(l) for l in lines if DEAL in l)
print(f"{len(lines)} records in the file; the one for {DEAL}:")
print(json.dumps(raw, indent=2))

source file: /Volumes/serverless_stable_s1omq8_catalog/seller_superapp/raw/v4/salesforce/Opportunity/batch=0000/part-0.json


15000 records in the file; the one for 0060g000000037YAAQ:
{
  "Id": "0060g000000037YAAQ",
  "Name": "Solis Construction - Vehicle Telematics + 1 New Business",
  "AccountId": "0010g0000000000AAA",
  "OwnerId": "0050g000000003OAAQ",
  "StageName": "Negotiation",
  "Amount": 173100,
  "CloseDate": "2026-12-30",
  "IsClosed": false,
  "IsWon": false,
  "Probability": 80,
  "ForecastCategoryName": "Commit",
  "Type": "New Business",
  "LeadSource": "Inbound",
  "Products__c": "Vehicle Telematics;Equipment Monitoring",
  "Primary_Product__c": "Vehicle Telematics",
  "Fleet_Size__c": 267,
  "Competitor__c": "RoadLynx",
  "Loss_Reason__c": null,
  "NextStep": "Put together a side-by-side comparison against roadlynx",
  "CreatedDate": "2026-06-13T16:00:00.000Z",
  "LastModifiedDate": "2026-09-03T21:00:00.000Z"
}


## 2. Bronze (streaming table, Auto Loader, append-only)

In [3]:
show(q("SELECT Id, Name, StageName, Amount, CloseDate, OwnerId, NextStep, batch, _ingested_at, _source_file FROM bronze_opportunity WHERE Id = :id ORDER BY batch", id=DEAL), transpose=True)

|              | 0                                                                                                              |
|:-------------|:---------------------------------------------------------------------------------------------------------------|
| Id           | 0060g000000037YAAQ                                                                                             |
| Name         | Solis Construction - Vehicle Telematics + 1 New Business                                                       |
| StageName    | Negotiation                                                                                                    |
| Amount       | 173100                                                                                                         |
| CloseDate    | 2026-12-30                                                                                                     |
| OwnerId      | 0050g000000003OAAQ                                                       

## 3. Silver (typed, deduplicated: one row per opportunity, latest `LastModifiedDate` wins)

In [4]:
show(q("SELECT opportunity_id, opportunity_name, stage_name, amount, close_date, owner_id, competitor, next_step, last_modified_date FROM silver_opportunities WHERE opportunity_id = :id", id=DEAL), transpose=True)

|                    | 0                                                        |
|:-------------------|:---------------------------------------------------------|
| opportunity_id     | 0060g000000037YAAQ                                       |
| opportunity_name   | Solis Construction - Vehicle Telematics + 1 New Business |
| stage_name         | Negotiation                                              |
| amount             | 173100.0                                                 |
| close_date         | 2026-12-30                                               |
| owner_id           | 0050g000000003OAAQ                                       |
| competitor         | RoadLynx                                                 |
| next_step          | Put together a side-by-side comparison against roadlynx  |
| last_modified_date | 2026-09-03T21:00:00.000Z                                 |


## 4. Gold: `gold_opportunity_pipeline` (joined with account, owner hierarchy, activity)
And the governed view the seller-facing tools read: `v_opportunity_pipeline` is row-filtered by `fn_can_see_owner`.

In [5]:
show(q("SELECT opportunity_id, account_name, owner_name, segment, stage, acv, fiscal_quarter, is_current_quarter, stage_age_days, days_since_last_activity, activities_30d FROM gold_opportunity_pipeline WHERE opportunity_id = :id", id=DEAL), transpose=True)
print()
print(q("SELECT count(*) AS rows_visible_to_me_in_v_opportunity_pipeline, count(DISTINCT owner_id) AS owners_visible FROM v_opportunity_pipeline").to_markdown(index=False))

|                          | 0                  |
|:-------------------------|:-------------------|
| opportunity_id           | 0060g000000037YAAQ |
| account_name             | Solis Construction |
| owner_name               | Jordan Rivera      |
| segment                  | Mid-Market         |
| stage                    | Negotiation        |
| acv                      | 173100.0           |
| fiscal_quarter           | FY27Q4             |
| is_current_quarter       | true               |
| stage_age_days           | 74                 |
| days_since_last_activity | 0                  |
| activities_30d           | 12                 |



|   rows_visible_to_me_in_v_opportunity_pipeline |   owners_visible |
|-----------------------------------------------:|-----------------:|
|                                             44 |                1 |


## 5. GenAI call insights (`ai_query` over the Meet transcripts, `gold_call_insights`)

In [6]:
ci = q("SELECT meeting_id, started_at, sentiment, competitors, objections, next_steps, summary, model, prompt_version FROM gold_call_insights WHERE opportunity_id = :id ORDER BY started_at DESC", id=DEAL)
print(f"{len(ci)} call(s) for this deal; most recent first")
for _, r in ci.head(3).iterrows():
    print("---"); print(r.to_markdown())

3 call(s) for this deal; most recent first
---
|                | 0                                                                                                                                                                                                                                                                                                                                                                                                                                       |
|:---------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| meeting_id     | njp-odhs-tuk                                

## 6. Deal score (OpenJev, `gold_deal_scores`): calibrated probabilities, not free text

In [7]:
show(q("SELECT opportunity_id, p_close_quarter, close_label, close_confidence, close_probs, next_action, next_action_conf, competitive_risk, churn_risk, model, latency_ms, input_tokens, scored_at FROM gold_deal_scores WHERE opportunity_id = :id", id=DEAL), transpose=True)

|                  | 0                                                                                                                                                                               |
|:-----------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| opportunity_id   | 0060g000000037YAAQ                                                                                                                                                              |
| p_close_quarter  | 0.8885589510035807                                                                                                                                                              |
| close_label      | Very likely                                                                                                                                                                     |
| clo

## 7. Next best actions (`gold_next_best_actions`, ranked, stable `action_id`)

In [8]:
show(q("SELECT action_id, action_type, source, title, why, priority_score, status, due_date FROM gold_next_best_actions WHERE opportunity_id = :id ORDER BY priority_score DESC", id=DEAL))

|    | action_id                                                        | action_type         | source   | title                                                                                       | why                                                                                                                |   priority_score | status   | due_date   |
|---:|:-----------------------------------------------------------------|:--------------------|:---------|:--------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------|-----------------:|:---------|:-----------|
|  0 | 0ff3a4f89937442923edfd51b1571e66f582169ecdfae9ba0000ed255a9275be | call_next_step      | call     | Loop in finance for a pricing review                                                        | Agreed on the 2026-11-16 call (owner: Dana Campbell (customer, CFO)), due 202

## 8. Lakebase Postgres row the app reads (`serving.opportunity_pipeline`, a synced table from `gold_opportunity_app`)

In [9]:
pg = connect(w, lb.Config(catalog=CATALOG, schema=SCHEMA).branch_path)
with pg.cursor() as c:
    c.execute("SELECT opportunity_id, opportunity_name, owner_name, stage, acv, close_date, p_close_quarter, close_label, next_action, competitive_risk, last_modified_date, scored_at FROM serving.opportunity_pipeline WHERE opportunity_id = %s", (DEAL,))
    print(pd.DataFrame(c.fetchall(), columns=[d.name for d in c.description]).T.to_markdown())
    t = time.perf_counter(); c.execute("SELECT stage FROM serving.opportunity_pipeline WHERE opportunity_id = %s", (DEAL,)); c.fetchone()
    print(f"\npoint lookup round trip from this client: {(time.perf_counter()-t)*1000:.1f} ms (includes laptop->AWS network)")
    c.execute("SELECT count(*) FROM serving.opportunity_pipeline"); pg_n = c.fetchone()[0]
uc_n = int(q("SELECT count(*) AS n FROM gold_opportunity_app").n[0])
print(f"row parity: UC gold_opportunity_app={uc_n}  Postgres serving.opportunity_pipeline={pg_n}  equal={uc_n == pg_n}")

|                    | 0                                                        |
|:-------------------|:---------------------------------------------------------|
| opportunity_id     | 0060g000000037YAAQ                                       |
| opportunity_name   | Solis Construction - Vehicle Telematics + 1 New Business |
| owner_name         | Jordan Rivera                                            |
| stage              | Negotiation                                              |
| acv                | 173100.0                                                 |
| close_date         | 2026-12-30                                               |
| p_close_quarter    | 0.8885589510035807                                       |
| close_label        | Very likely                                              |
| next_action        | address_risk                                             |
| competitive_risk   | 0.8175744584140004                                       |
| last_modified_

row parity: UC gold_opportunity_app=15000  Postgres serving.opportunity_pipeline=15000  equal=True


## 9. Genie answer ("Fleetwise Revenue Copilot", on-behalf-of the caller, row-scoped views only)

In [10]:
from genie.benchmark import ask
space = (ROOT / "src/genie/space_id.txt").read_text().strip()
a = ask(w, space, "Show my top 5 deals by ACV times close probability.")
print("status:", a["status"], f"latency {a['latency_s']:.1f} s")
print("generated SQL:\n" + (a["sql"] or ""))
print(pd.DataFrame(a["all_rows"], columns=a["columns"]).to_markdown(index=False))
print("\nour deal in Genie's answer:", any(DEAL in map(str, r) for r in a["all_rows"]))

status: COMPLETED latency 20.6 s
generated SQL:
SELECT
  `v_opportunity_pipeline`.`opportunity_id`,
  `v_opportunity_pipeline`.`opportunity_name`,
  `v_opportunity_pipeline`.`account_name`,
  `v_opportunity_pipeline`.`stage`,
  `v_opportunity_pipeline`.`acv`,
  ROUND(`v_deal_scores`.`p_close_quarter`, 2) AS `p_close_quarter`,
  ROUND(`v_opportunity_pipeline`.`acv` * `v_deal_scores`.`p_close_quarter`, 2) AS `weighted_acv`,
  `v_opportunity_pipeline`.`close_date`
FROM `serverless_stable_s1omq8_catalog`.`seller_superapp`.`v_opportunity_pipeline` AS `v_opportunity_pipeline`
INNER JOIN `serverless_stable_s1omq8_catalog`.`seller_superapp`.`v_deal_scores` AS `v_deal_scores`
  ON `v_opportunity_pipeline`.`opportunity_id` = `v_deal_scores`.`opportunity_id`
WHERE `v_opportunity_pipeline`.`is_closed` = false
  AND `v_opportunity_pipeline`.`acv` IS NOT NULL
  AND `v_deal_scores`.`p_close_quarter` IS NOT NULL
ORDER BY (`v_opportunity_pipeline`.`acv` * `v_deal_scores`.`p_close_quarter`) DESC,
  `v_o

## 10. The app's REST API (deployed Databricks App, user OAuth, persona = Jordan Rivera)

In [11]:
app = w.apps.get("fleetwise-seller-hub")
print("app:", app.name, app.app_status.state.value, app.url)
s = requests.Session(); s.headers["Authorization"] = f"Bearer {w.config.oauth_token().access_token}"
me = s.get(app.url + "/api/me", timeout=120).json()
print(json.dumps({k: me[k] for k in ("persona_name", "role", "visible_owner_count") if k in me}, indent=2))
home = s.get(app.url + "/api/home", timeout=120).json()
print("freshness_ts:", home["freshness_ts"], "| fiscal_quarter:", home["fiscal_quarter"])
print("KPIs:", json.dumps(home["kpis"]))
print(pd.DataFrame(home["priority_deals"])[["opportunity_id", "opportunity_name", "stage", "acv", "p_close_quarter"]].head(5).to_markdown(index=False))
deal = s.get(app.url + f"/api/deals/{DEAL}", timeout=120).json()
print("\nGET /api/deals/" + DEAL + " (excerpt):")
print(json.dumps({"opportunity": {k: deal["opportunity"][k] for k in ("opportunity_name", "stage", "acv", "owner_name")},
                  "scores": deal["scores"], "insights": len(deal["insights"]), "actions": [x.get("title") for x in deal["actions"]][:5],
                  "contacts_sample": deal["contacts"][:2]}, indent=2, default=str))

app: fleetwise-seller-hub RUNNING https://fleetwise-seller-hub-7474646802937769.aws.databricksapps.com


{
  "persona_name": "Jordan Rivera",
  "role": "seller",
  "visible_owner_count": 1
}


freshness_ts: 2026-11-16T14:00:00.000Z | fiscal_quarter: FY27Q4
KPIs: {"fiscal_quarter": "FY27Q4", "reps": 1, "quota": 500000, "closed_won_acv": 176700, "attainment_pct": 0.3534, "open_pipeline_acv": 1131600, "commit_acv": 401100, "best_case_acv": 565700, "open_opportunities": 18, "won_opportunities": 1, "coverage_ratio": 3.5001546551190845}
| opportunity_id     | opportunity_name                                               | stage       |    acv |   p_close_quarter |
|:-------------------|:---------------------------------------------------------------|:------------|-------:|------------------:|
| 0060g000000037YAAQ | Solis Construction - Vehicle Telematics + 1 New Business       | Negotiation | 173100 |          0.888559 |
| 0060g000000037hAAA | Reid General Contractors - Vehicle Telematics + 1 New Business | Proposal    | 105800 |          0.904395 |
| 0060g000000037kAAA | Jones Fire Protection - Vehicle Telematics Renewal             | Demo        | 205900 |          0.456997 |
|

## Summary
The same `opportunity_id` appears in the raw file, bronze, silver, gold, the AI tables, the next-best-action
queue, the Lakebase row, Genie's answer and the app API, so every layer is connected by one key and one
governed lineage graph (see `evidence/10_e2e.md` for the `system.access.table_lineage` edges).
Score quality is in `evidence/06_nba_and_eval.md` (AUC 0.695, top-decile lift 2.06x).